In [33]:
import argparse
import torch
import random
import sys
import gc
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset, concatenate_datasets
from span_marker import SpanMarkerModel
from gliner import GLiNER

# Import path resolution logic
from EXPERIMENTS.finetune import get_output_dir
# from dataset_processing import shorten_name

# --- CONFIGURATION ---
DEFAULT_SEEDS = [0, 42, 3012, 33, 131]

all_tags = """Asset
Body of Water
Body Part
Chemical
Disease
Ecosystem
Energy Source
Field of Study
Geographical Feature
Intellectual Artefact
Location
Mathematical Expression
Measuring Device
Meteorological Phenomenon
Method
Natural Disaster
Natural Phenomenon
Organism
Organization
Other
Person
Physical Artefact
Physical Phenomenon
Policy or Objective
Quantity
Satellite
System
Time Period"""

TARGET_TAGS = all_tags.split("\n")

# --- THRESHOLDS (The 4-1 Rule) ---
SUCCESS_THRESHOLD = 3  # Must be found in >= 4 seeds
FAILURE_THRESHOLD = 2  # Must be found in <= 1 seed

def load_gold_data(dataset_id):
    print(f"--- Loading Dataset: {dataset_id} ---")
    ds = load_dataset(dataset_id)
    splits = [ds[split] for split in ds.keys()]
    merged = concatenate_datasets(splits)
    features = merged.features["ner_tags"].feature.names
    id2label = {i: name for i, name in enumerate(features)}
    return merged, id2label

def extract_gold_spans(row, id2label):
    tokens = row['tokens']
    tags = row['ner_tags']
    extracted_gold = []
    curr_start, curr_label = None, None
    for i, tag_id in enumerate(tags):
        tag_name = id2label[tag_id]
        if tag_name.startswith("B-"):
            if curr_label: extracted_gold.append((" ".join(tokens[curr_start:i]), curr_label))
            curr_start, curr_label = i, tag_name[2:]
        elif tag_name.startswith("I-") and curr_label == tag_name[2:]: continue
        else:
            if curr_label: extracted_gold.append((" ".join(tokens[curr_start:i]), curr_label))
            curr_label, curr_start = None, None
    if curr_label: extracted_gold.append((" ".join(tokens[curr_start:]), curr_label))
    return set(extracted_gold)

def run_inference_single_seed(model_path, model_type, texts, device):
    predictions = []
    if model_type == "SPANMARKER":
        model = SpanMarkerModel.from_pretrained(model_path)
        if device.type == 'cuda': model.cuda()
        output = model.predict(texts, show_progress_bar=False)
        for sent_preds in output:
            predictions.append({(ent['span'], ent['label']) for ent in sent_preds})
        del model
    elif model_type == "GLINER":
        model = GLiNER.from_pretrained(model_path)
        model.to(device)
        from dataset_processing import CLIRENER_LABELS_V1
        labels = list(CLIRENER_LABELS_V1)
        for text in texts:
            out = model.predict_entities(text, labels, threshold=0.5)
            predictions.append({(ent['text'], ent['label']) for ent in out})
        del model
    torch.cuda.empty_cache()
    return predictions

def get_seed_frequencies(base_dir, model_type, model_id, train_dataset, texts, device, seeds):
    print(f"\n>>> Analyzing Model: {model_id}")
    freq_maps = [Counter() for _ in texts]
    for seed in seeds:
        path = get_output_dir(base_dir, model_type, model_id, train_dataset, seed) / "checkpoint-final"
        if not path.exists(): continue
        print(f"  - Processing Seed {seed}...")
        seed_preds = run_inference_single_seed(str(path), model_type, texts, device)
        for idx, sent_set in enumerate(seed_preds):
            freq_maps[idx].update(sent_set)
        gc.collect()
    return freq_maps

def format_context(text, entity_text):
    start = text.find(entity_text)
    if start == -1: return f"**{entity_text}**"
    end = start + len(entity_text)
    ctx_start, ctx_end = max(0, start - 35), min(len(text), end + 35)
    snippet = text[ctx_start:ctx_end]
    formatted = snippet.replace(entity_text, f"**{entity_text}**")
    return f"{'...' if ctx_start > 0 else ''}{formatted}{'...' if ctx_end < len(text) else ''}"




BASELINE = "FacebookAI/roberta-base"
CHALLENGER = "nasa-impact/nasa-smd-ibm-v0.1"

BASE_T = "SPANMARKER"
CHAL_T = "SPANMARKER"

TRAIN_DATASET = "P0L3/CliReNER_v_1_1_28_SILVER"
EVAL_DATASET = "P0L3/CliReNER_v_1_1_28_GOLD_authorannots"


# 1. Load Data
dataset, id2label = load_gold_data(EVAL_DATASET)
texts, gold_sets = dataset['text'], [extract_gold_spans(row, id2label) for row in dataset]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# 2. Get frequencies
base_freqs = get_seed_frequencies("EXPERIMENTS/models", BASE_T, BASELINE, TRAIN_DATASET, texts, device, DEFAULT_SEEDS)
chall_freqs = get_seed_frequencies("EXPERIMENTS/models", CHAL_T, CHALLENGER, TRAIN_DATASET, texts, device, DEFAULT_SEEDS)

# 3. Categorization
results = {tag: {"lost": [], "shared": [], "gained": []} for tag in TARGET_TAGS}

print(f"\n--- Categorizing (4-1 Rule: Success >={SUCCESS_THRESHOLD}, Failure <={FAILURE_THRESHOLD}) ---")
    
for i, text in enumerate(texts):
    G = gold_sets[i]
    B_map, C_map = base_freqs[i], chall_freqs[i]

    for (ent_text, ent_label) in G:
        if ent_label not in TARGET_TAGS: continue
        
        b_votes = B_map.get((ent_text, ent_label), 0)
        c_votes = C_map.get((ent_text, ent_label), 0)
        
        ctx = format_context(text, ent_text)

        if b_votes >= SUCCESS_THRESHOLD and c_votes >= SUCCESS_THRESHOLD:
            results[ent_label]["shared"].append(ctx)
        elif b_votes >= SUCCESS_THRESHOLD and c_votes <= FAILURE_THRESHOLD:
            results[ent_label]["lost"].append(ctx)
        elif c_votes >= SUCCESS_THRESHOLD and b_votes <= FAILURE_THRESHOLD:
            results[ent_label]["gained"].append(ctx)



# with open(output_path, "w", encoding="utf-8") as f:
#     f.write(f"# Robust Architectural Comparison: {BASELINE} vs {CHALLENGER}\n")
#     f.write(f"- **Strictness (4-1 Rule):**\n")
#     f.write(f"  - **Stable Success:** Found in $\ge {SUCCESS_THRESHOLD}/5$ seeds.\n")
#     f.write(f"  - **Stable Failure:** Found in $\le {FAILURE_THRESHOLD}/5$ seeds.\n\n")
    
#     f.write(f"| Entity Type | Correct in Baseline ONLY ({base_short}) | Correct in BOTH | Correct in Challenger ONLY ({chall_short}) |\n")
#     f.write("|---|---|---|---|\n")
    
#     for tag in TARGET_TAGS:
#         def sample_cell(key):
#             items = list(set(results[tag][key]))
#             if not items: return "*(None found matching 4-1 rule)*"
#             return "<br><br>".join(random.sample(items, min(2, len(items))))
        
#         f.write(f"| **{tag}** | {sample_cell('lost')} | {sample_cell('shared')} | {sample_cell('gained')} |\n")

print("Done.")



--- Loading Dataset: P0L3/CliReNER_v_1_1_28_GOLD_authorannots ---

>>> Analyzing Model: FacebookAI/roberta-base
  - Processing Seed 0...
  - Processing Seed 42...
  - Processing Seed 3012...
  - Processing Seed 33...
  - Processing Seed 131...

>>> Analyzing Model: nasa-impact/nasa-smd-ibm-v0.1
  - Processing Seed 0...
  - Processing Seed 42...
  - Processing Seed 3012...
  - Processing Seed 33...
  - Processing Seed 131...

--- Categorizing (4-1 Rule: Success >=3, Failure <=2) ---
Done.


In [41]:
for i,text in enumerate(texts):
    if "vegetation growth" in text:
        print(i, text)

83 Unlike relative humidity (RH), VPD provides a linear measure of the exponential relationship between temperature and evapotranspiration (Anderson, 1936), and increasing VPD around the globe has been linked to reduced vegetation growth (Yuan et al., 2019) and wildfires (Abatzoglou et al., 2018b).


In [42]:
chall_freqs[83]

Counter({('VPD', 'Quantity'): 5,
         ('globe', 'Location'): 5,
         ('RH', 'Quantity'): 5,
         ('evapotranspiration', 'Quantity'): 5,
         ('temperature', 'Quantity'): 5,
         ('2018b', 'Time Period'): 5,
         ('relative humidity', 'Quantity'): 5,
         ('2019', 'Time Period'): 5,
         ('linear measure', 'Quantity'): 4,
         ('wildfires', 'Natural Disaster'): 3,
         ('exponential relationship', 'Physical Phenomenon'): 2,
         ('wildfires', 'Meteorological Phenomenon'): 2,
         ('exponential relationship', 'Quantity'): 2,
         ('Abatzoglou et al.,', 'Person'): 1,
         ('exponential relationship', 'Other'): 1,
         ('linear measure', 'Other'): 1,
         ('vegetation growth', 'Organism'): 1,
         ('1936', 'Time Period'): 1,
         ('vegetation growth', 'Natural Phenomenon'): 1,
         ('vegetation growth', 'Meteorological Phenomenon'): 1,
         ('increasing', 'Quantity'): 1,
         ('vegetation growth', 'Physical

In [35]:
results["Natural Phenomenon"]

{'lost': ['...s for acclimation and facilitating **compensatory growth** following drought.',
  '...ay a role in drought tolerance and **drought avoidance** by supplying N to leaves for accli...',
  '...en (N) fixation may play a role in **drought tolerance** and drought avoidance by supplying...',
  '...es linking climate variability and **disease transmission** across the various agro-ecological...',
  '...e globe has been linked to reduced **vegetation growth** (Yuan et al.,\xa02019) and wildfires ...',
  'Species-level analysis of **vegetation patterns** enables quantifying and monitoring...',
  'Therefore, when modeling **vegetation patterns**, ecologists must carefully select ...',
  '...information required to assess the **triggering phenomena**.',
  '...sult in the rapid growth of algae, **eutrophication**, and the death of animal life due ...'],
 'shared': [],
 'gained': []}

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD_authorannots ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192
Error: File RESULTS/LLM_PREDICTIONS/ner_results_gemini_2_5_pro.jsonl not found.
Aligned 0 rows. (Skipped: 192 missing, 0 length mismatch)
No errors found for entity type: Natural Phenomenon

--- Overall Performance (Strict Match) ---


TypeError: tuple indices must be integers or slices, not str

In [1]:
from span_marker import SpanMarkerModel

# Download from the 🤗 Hub
model = SpanMarkerModel.from_pretrained("P0L3/CliReNER-indus-sde-v0.2")

# Run inference
text = "As shown in these specialised domains, general-domain, off-the-shelf NER systems often fail to capture domain-specific terminology, whereas models adapted or fine-tuned on in-domain corpora consistently achieve superior performance (Lee et al. 2019; Gururangan et al. 2020)."
entities = model.predict(text)

for entity in entities:
    print(f"Entity: {entity['span']} | Label: {entity['label']} | Score: {entity['score']:.4f}")


model.safetensors:   0%|          | 0.00/500M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

SpanMarker model predictions are being computed on the CPU while CUDA is available. Moving the model to CUDA using `model.cuda()` before performing predictions is heavily recommended to significantly boost prediction speeds.


Entity: specialised domains | Label: Other | Score: 0.4236
Entity: general-domain | Label: Other | Score: 0.6351
Entity: off-the-shelf | Label: Other | Score: 0.3454
Entity: NER systems | Label: Method | Score: 0.5303
Entity: domain-specific terminology | Label: Other | Score: 0.4407
Entity: models | Label: Method | Score: 0.8604
Entity: in-domain corpora | Label: Intellectual Artefact | Score: 0.8342
Entity: Lee et al. | Label: Person | Score: 0.7713
Entity: Gururangan et al. | Label: Person | Score: 0.7211
Entity: 2020 | Label: Time Period | Score: 0.9387


In [5]:
from gliner import GLiNER

# Load model
model = GLiNER.from_pretrained(
    "P0L3/CliReNER-gliner_medium-v2.5",
    load_tokenizer=True
)

# Input text
text = """
In recent years, climate NLP has seen its nascent with the introduction of domain-specific models such as ClimateBERT (Webersinke et al. 2022), ClimateGPT (Thulke et al. 2024), and CliReBERT (Poleksić and Martinčić-Ipšić 2025), alongside related efforts (Bhattacharjee et al. 2024; Schimanski et al. 2024)
"""

# Labels
labels = [
    "Ecosystem", "Energy Source", "Natural Disaster", 
    "Meteorological Phenomenon", "Quantity", "Intellectual Artefact", 
    "Body of Water", "Disease", "Location", 
    "Physical Phenomenon", "Chemical", "Time Period", 
    "Organization", "Natural Phenomenon", "Field of Study", 
    "Mathematical Expression", "Measuring Device", "Geographical Feature", 
    "System", "Satellite", "Organism", 
    "Method", "Other", "Person", 
    "Physical Artefact", "Body Part", "Asset",
    "Policy",
]

# Predict entities
entities = model.predict_entities(text, labels)

# Match SpanMarker-style output
for entity in entities:
    span = entity["text"]
    label = entity["label"]
    score = entity["score"]  # fallback if score not present

    print(f"Entity: {span} | Label: {label} | Score: {score:.4f}")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

added_tokens.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

gliner_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/834M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Entity: recent years | Label: Time Period | Score: 0.8569
Entity: climate NLP | Label: Method | Score: 0.6270
Entity: domain-specific models | Label: Method | Score: 0.7560
Entity: ClimateBERT | Label: Method | Score: 0.8053
Entity: Webersinke et al. | Label: Person | Score: 0.7729
Entity: 2022 | Label: Time Period | Score: 0.8569
Entity: ClimateGPT | Label: Method | Score: 0.7935
Entity: Thulke et al. | Label: Person | Score: 0.7884
Entity: 2024 | Label: Time Period | Score: 0.8953
Entity: CliReBERT | Label: Method | Score: 0.8261
Entity: Poleksić and Martinčić-Ipšić | Label: Person | Score: 0.8301
Entity: 2025 | Label: Time Period | Score: 0.9154
Entity: related efforts | Label: Other | Score: 0.5821
Entity: Bhattacharjee et al. | Label: Person | Score: 0.7508
Entity: 2024 | Label: Time Period | Score: 0.8572
Entity: Schimanski et al. | Label: Person | Score: 0.7806
Entity: 2024 | Label: Time Period | Score: 0.9111


## Data Preview

In [17]:
from EXPERIMENTS.evaluate_gold import load_and_merge_gold_data
from dataset_processing import hf_dataset_to_gliner_format

data = load_and_merge_gold_data("P0L3/CliReNER_v_1_1_28_GOLD")

data_ner = hf_dataset_to_gliner_format(data[0]["test"], data[1])

def pretty_print_ner(row):
    """
    Reconstructs the tokenized sentence and injects Markdown-style 
    entity annotations: [entity text](LABEL)
    """
    tokens = row['tokenized_text']
    # Ensure spans are sorted by start index to prevent alignment issues
    spans = sorted(row['ner'], key=lambda x: x[0])
    
    output_parts = []
    current_idx = 0
    
    for span in spans:
        start, end, label = span[0], span[1], span[2]
        
        # 1. Add preceding non-entity tokens
        if start > current_idx:
            output_parts.extend(tokens[current_idx:start])
            
        # 2. Extract and format the entity tokens (inclusive boundary slicing)
        entity_tokens = tokens[start:end + 1]
        entity_text = " ".join(entity_tokens)
        formatted_entity = f"[{entity_text}]({label})"
        output_parts.append(formatted_entity)
        
        # 3. Advance the pointer past the current entity
        current_idx = end + 1
        
    # 4. Add any remaining trailing tokens
    if current_idx < len(tokens):
        output_parts.extend(tokens[current_idx:])
        
    # Join with standard spacing for tokenized streams
    formatted_sentence = " ".join(output_parts)
    
    # Optional: Clean up spaces before punctuation (e.g., " .", " ,", " ( ")
    # to make the output look more like a natural sentence.
    formatted_sentence = formatted_sentence.replace(" .", ".").replace(" ,", ",").replace(" ( ", " (")
    
    return formatted_sentence

--- Loading and Merging GOLD Dataset: P0L3/CliReNER_v_1_1_28_GOLD ---
Merged ['train', 'validation', 'test'] into a single dataset of size: 192


In [19]:
for row in data_ner:
    # print(f"ID: {row['id']}")
    print(pretty_print_ner(row))
    # print("-" * 80)

Excluding [time points](Time Period) when [IM PB](Chemical) was administered within [7 days](Time Period) prior to [sampling](Method) ([n = 11](Mathematical Expression) ), peak samples collected [< 4 h](Time Period) after [dosing](Method) ([n = 3](Mathematical Expression) ), and trough samples collected [> 28 h](Time Period) after [dosing](Method) ([n = 14](Mathematical Expression) ), [thirty-five](Quantity) time points were trough samples collected [24 h](Time Period) after [dosing](Method).
The [EASM index](Quantity) of the [four-member ensemble averaged solar-only forcing experiment](Method) is significantly correlated with the [solar activity](Physical Phenomenon) with a [correlation coefficient](Mathematical Expression) of [0. 414](Quantity) (the [effective sample size](Quantity) [n](Quantity) [68](Quantity), [p](Quantity) [0. 05](Quantity) based on the [Student ’ s t test](Method) ) during the [strong cycle epoch](Time Period) ([900 – 1285](Time Period) ), whereas there is no [st